# Probador de Modelos CNN

Cargue su modelo entrenado (.h5) y pruébelo con imágenes.

**Usos:**
- Después de Lab 3: verificar que su modelo funcione antes de probar con cámara
- Proyecto final: evaluar y comparar modelos entrenados

> No entrena modelos — solo los carga y prueba.

In [ ]:
import os
import glob
import random
import io
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array, ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import ipywidgets as widgets
from IPython.display import display, clear_output

random.seed(42)
np.random.seed(42)

# Estado global
_state = {'model': None, 'class_names': [], 'img_size': 128}

print(f"TensorFlow {tf.__version__} — OK")

---
## Paso 1: Cargar Modelo

Suba su archivo `.h5` o ingrese la ruta. Luego configure las clases y tamaño de imagen.

In [ ]:
# --- Widget: Cargar modelo ---
w_upload = widgets.FileUpload(accept='.h5,.keras', multiple=False,
                              description='Subir .h5')
w_path = widgets.Text(value='', placeholder='O escriba la ruta al .h5',
                      description='Ruta:', layout=widgets.Layout(width='400px'))
w_classes = widgets.Text(value='', placeholder='rojo, azul, verde',
                         description='Clases:', layout=widgets.Layout(width='400px'))
w_imgsize = widgets.Dropdown(options=[64, 128, 224], value=128,
                             description='IMG_SIZE:')
w_btn_load = widgets.Button(description='Cargar modelo', button_style='primary',
                            icon='upload')
w_out_load = widgets.Output()

def _cargar_modelo(_):
    with w_out_load:
        clear_output(wait=True)

        # Validar clases
        raw = w_classes.value.strip()
        if not raw:
            print("\u2718 Ingrese los nombres de las clases separados por comas.")
            return
        class_names = [c.strip() for c in raw.split(',') if c.strip()]
        if len(class_names) < 2:
            print("\u2718 Necesita al menos 2 clases.")
            return

        # Obtener ruta del modelo
        model_path = None
        if w_upload.value:
            fname = list(w_upload.value.keys())[0] if isinstance(w_upload.value, dict) else w_upload.value[0].name
            content = list(w_upload.value.values())[0]['content'] if isinstance(w_upload.value, dict) else w_upload.value[0].content
            model_path = f'/tmp/{fname}'
            with open(model_path, 'wb') as f:
                f.write(content)
            print(f"Archivo subido: {fname}")
        elif w_path.value.strip():
            model_path = w_path.value.strip()
        else:
            print("\u2718 Suba un archivo .h5 o ingrese la ruta.")
            return

        if not os.path.exists(model_path):
            print(f"\u2718 No se encontr\u00f3 el archivo: {model_path}")
            return

        # Cargar
        try:
            model = load_model(model_path, compile=False)
            model.compile(optimizer='adam', loss='categorical_crossentropy',
                          metrics=['accuracy'])
        except Exception as e:
            print(f"\u2718 Error al cargar modelo: {e}")
            return

        # Verificar compatibilidad
        img_size = w_imgsize.value
        expected_shape = (None, img_size, img_size, 3)
        actual_shape = model.input_shape
        if actual_shape[1:3] != (img_size, img_size):
            print(f"\u26a0 El modelo espera {actual_shape[1]}\u00d7{actual_shape[2]} "
                  f"pero seleccion\u00f3 IMG_SIZE={img_size}")
            print(f"  Ajustando IMG_SIZE a {actual_shape[1]}")
            img_size = actual_shape[1]
            w_imgsize.value = img_size

        n_outputs = model.output_shape[-1]
        if n_outputs != len(class_names):
            print(f"\u26a0 El modelo tiene {n_outputs} salidas pero se dieron "
                  f"{len(class_names)} clases.")

        # Guardar estado
        _state['model'] = model
        _state['class_names'] = class_names
        _state['img_size'] = img_size

        print(f"\n\u2714 Modelo cargado: {os.path.basename(model_path)}")
        print(f"  Clases: {class_names}")
        print(f"  IMG_SIZE: {img_size}")
        print(f"  Par\u00e1metros: {model.count_params():,}")
        print()
        model.summary()

w_btn_load.on_click(_cargar_modelo)

display(widgets.VBox([
    widgets.HTML('<b>Opci\u00f3n A:</b> Subir archivo'),
    w_upload,
    widgets.HTML('<b>Opci\u00f3n B:</b> Escribir ruta'),
    w_path,
    widgets.HTML('<hr><b>Configuraci\u00f3n:</b>'),
    w_classes,
    w_imgsize,
    w_btn_load,
    w_out_load
]))

---
## Paso 2: Evaluar con dataset

Evalúe el modelo sobre un directorio de imágenes organizadas en subcarpetas por clase.

In [ ]:
# --- Widget: Evaluaci\u00f3n sobre dataset ---
w_dataset_path = widgets.Text(
    value='../data/tapitas',
    description='Dataset:', layout=widgets.Layout(width='400px'),
    placeholder='Ruta al directorio con subcarpetas por clase'
)
w_btn_eval = widgets.Button(description='Evaluar', button_style='success',
                            icon='check')
w_out_eval = widgets.Output()

def _evaluar(_):
    with w_out_eval:
        clear_output(wait=True)
        model = _state['model']
        class_names = _state['class_names']
        img_size = _state['img_size']

        if model is None:
            print("\u2718 Primero cargue un modelo (Paso 1).")
            return

        dataset_dir = w_dataset_path.value.strip()
        if not os.path.isdir(dataset_dir):
            print(f"\u2718 No se encontr\u00f3 el directorio: {dataset_dir}")
            return

        # Crear generador
        datagen = ImageDataGenerator(rescale=1./255)
        try:
            generator = datagen.flow_from_directory(
                dataset_dir,
                target_size=(img_size, img_size),
                batch_size=32,
                class_mode='categorical',
                shuffle=False,
                classes=class_names
            )
        except Exception as e:
            print(f"\u2718 Error al leer dataset: {e}")
            print(f"  Verifique que existan subcarpetas: {class_names}")
            return

        if generator.samples == 0:
            print("\u2718 No se encontraron im\u00e1genes en el directorio.")
            return

        # Evaluar
        print(f"Evaluando {generator.samples} im\u00e1genes...")
        loss, acc = model.evaluate(generator, verbose=0)
        print(f"\nAccuracy: {acc:.1%}")
        print(f"Loss: {loss:.4f}")

        # Predicciones
        generator.reset()
        y_pred_probs = model.predict(generator, verbose=1)
        y_pred = np.argmax(y_pred_probs, axis=1)
        y_true = generator.classes
        labels = list(generator.class_indices.keys())

        # Classification report
        print("\nReporte de Clasificaci\u00f3n:")
        print("=" * 60)
        print(classification_report(y_true, y_pred, target_names=labels,
                                    zero_division=0))

        # Confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        n = len(labels)
        plt.figure(figsize=(max(6, n * 1.5), max(5, n * 1.2)))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=labels, yticklabels=labels)
        plt.xlabel('Predicci\u00f3n')
        plt.ylabel('Real')
        plt.title(f'Matriz de Confusi\u00f3n (Accuracy: {acc:.1%})')
        plt.tight_layout()
        plt.show()

        # Mostrar errores
        errores = []
        for i in range(len(y_true)):
            if y_pred[i] != y_true[i]:
                errores.append({
                    'file': generator.filenames[i],
                    'real': labels[y_true[i]],
                    'pred': labels[y_pred[i]],
                    'conf': y_pred_probs[i][y_pred[i]]
                })

        if errores:
            n_show = min(8, len(errores))
            print(f"\nIm\u00e1genes mal clasificadas: {len(errores)} de {len(y_true)}")
            fig, axes = plt.subplots(2, 4, figsize=(16, 8))
            for ax in axes.ravel():
                ax.axis('off')
            for i, ax in enumerate(axes.ravel()):
                if i >= n_show:
                    break
                err = errores[i]
                img_path = os.path.join(dataset_dir, err['file'])
                if os.path.exists(img_path):
                    img = load_img(img_path, target_size=(img_size, img_size))
                    ax.imshow(img)
                    ax.set_title(f"Real: {err['real']}\nPred: {err['pred']} "
                                 f"({err['conf']:.0%})",
                                 fontsize=10, color='red')
            plt.suptitle('Im\u00e1genes MAL clasificadas', fontsize=14)
            plt.tight_layout()
            plt.show()
        else:
            print("\n\u2714 No hay errores \u2014 clasificaci\u00f3n perfecta.")

w_btn_eval.on_click(_evaluar)

display(widgets.VBox([
    w_dataset_path,
    w_btn_eval,
    w_out_eval
]))

---
## Paso 3: Probar con imágenes

Pruebe el modelo con imágenes individuales: una aleatoria del dataset o una foto nueva.

In [ ]:
# --- Funciones auxiliares ---
def _predecir_y_mostrar(img_path=None, img_array_raw=None, titulo=''):
    """Predice y muestra resultado con barras de confianza."""
    model = _state['model']
    class_names = _state['class_names']
    img_size = _state['img_size']

    if img_path:
        img = load_img(img_path, target_size=(img_size, img_size))
        img_array = img_to_array(img) / 255.0
        img_display = img
    elif img_array_raw is not None:
        from PIL import Image
        pil_img = Image.open(io.BytesIO(img_array_raw)).resize((img_size, img_size))
        img_array = np.array(pil_img).astype('float32') / 255.0
        if img_array.ndim == 2:  # grayscale
            img_array = np.stack([img_array] * 3, axis=-1)
        elif img_array.shape[-1] == 4:  # RGBA
            img_array = img_array[:, :, :3]
        img_display = pil_img
    else:
        return

    batch = np.expand_dims(img_array, 0)
    predictions = model.predict(batch, verbose=0)[0]
    pred_class = np.argmax(predictions)
    confidence = predictions[pred_class]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4),
                                    gridspec_kw={'width_ratios': [1, 1.2]})

    ax1.imshow(img_display)
    color = 'green' if confidence > 0.7 else 'orange'
    ax1.set_title(f'{class_names[pred_class]} ({confidence:.1%})',
                  fontsize=14, color=color, fontweight='bold')
    ax1.axis('off')

    colors = ['#2ecc71' if i == pred_class else '#bdc3c7'
              for i in range(len(class_names))]
    bars = ax2.barh(class_names, predictions, color=colors)
    ax2.set_xlim(0, 1)
    ax2.set_xlabel('Confianza')
    for bar, val in zip(bars, predictions):
        ax2.text(bar.get_width() + 0.02,
                 bar.get_y() + bar.get_height() / 2,
                 f'{val:.1%}', va='center', fontsize=10)

    if titulo:
        fig.suptitle(titulo, fontsize=11)
    plt.tight_layout()
    plt.show()
    return class_names[pred_class], confidence

In [ ]:
# --- Widget: Predicci\u00f3n individual ---
w_btn_random = widgets.Button(description='Imagen aleatoria del dataset',
                              button_style='info', icon='random')
w_upload_img = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False,
                                  description='Subir foto')
w_btn_predict = widgets.Button(description='Predecir foto subida',
                               button_style='warning', icon='eye')
w_out_pred = widgets.Output()

def _imagen_aleatoria(_):
    with w_out_pred:
        clear_output(wait=True)
        if _state['model'] is None:
            print("\u2718 Primero cargue un modelo (Paso 1).")
            return

        dataset_dir = w_dataset_path.value.strip()
        if not os.path.isdir(dataset_dir):
            print(f"\u2718 No se encontr\u00f3: {dataset_dir}")
            return

        imgs = []
        for ext in ('*.jpg', '*.jpeg', '*.png'):
            imgs += glob.glob(os.path.join(dataset_dir, '**', ext), recursive=True)
        if not imgs:
            print("No se encontraron im\u00e1genes.")
            return

        img_path = random.choice(imgs)
        clase_real = os.path.basename(os.path.dirname(img_path))
        pred, conf = _predecir_y_mostrar(img_path=img_path,
                                          titulo=f'Clase real: {clase_real}')
        ok = '\u2714' if pred == clase_real else '\u2718'
        print(f"{ok} Real: {clase_real} | Predicci\u00f3n: {pred} ({conf:.1%})")

def _predecir_subida(_):
    with w_out_pred:
        clear_output(wait=True)
        if _state['model'] is None:
            print("\u2718 Primero cargue un modelo (Paso 1).")
            return
        if not w_upload_img.value:
            print("\u2718 Suba una imagen primero.")
            return

        if isinstance(w_upload_img.value, dict):
            content = list(w_upload_img.value.values())[0]['content']
        else:
            content = w_upload_img.value[0].content

        pred, conf = _predecir_y_mostrar(img_array_raw=content,
                                          titulo='Imagen subida')
        print(f"Predicci\u00f3n: {pred} ({conf:.1%})")

w_btn_random.on_click(_imagen_aleatoria)
w_btn_predict.on_click(_predecir_subida)

display(widgets.VBox([
    widgets.HBox([w_btn_random]),
    widgets.HTML('<hr>'),
    widgets.HBox([w_upload_img, w_btn_predict]),
    w_out_pred
]))

---
## Paso 4: Configuración para cámara

Genere el bloque de configuración listo para copiar en `scripts/prueba_video.py`.

In [ ]:
# --- Widget: Exportar configuraci\u00f3n ---
w_btn_config = widgets.Button(description='Generar configuraci\u00f3n',
                              button_style='success', icon='cog')
w_out_config = widgets.Output()

def _generar_config(_):
    with w_out_config:
        clear_output(wait=True)
        if _state['model'] is None:
            print("\u2718 Primero cargue un modelo (Paso 1).")
            return

        class_names = _state['class_names']
        img_size = _state['img_size']

        print("Copie este bloque en scripts/prueba_video.py:")
        print("=" * 50)
        print(f'MODEL_PATH = "labs/mi_modelo.h5"')
        print(f'IMG_SIZE = {img_size}')
        print(f'CLASS_NAMES = {class_names}')
        print(f'CONFIDENCE_THRESHOLD = 0.6')
        print("=" * 50)
        print()
        print("Recuerde:")
        print("  1. Copie su .h5 a la carpeta labs/ (o ajuste MODEL_PATH)")
        print("  2. El celular debe estar en la MISMA posici\u00f3n que al grabar")
        print("  3. La iluminaci\u00f3n debe ser similar")

w_btn_config.on_click(_generar_config)

display(widgets.VBox([
    w_btn_config,
    w_out_config
]))